In [89]:
## Required libraries
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import time

In [90]:
#base_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])
#raw_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\raw_data_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])

base_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])
#raw_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\raw_data_ex.csv', encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])


In [93]:
# Required functions

def preprocess(x):
    
    x["Arrival"]= pd.to_datetime(x["Arrival"]) 
    x["Departure"]= pd.to_datetime(x["Departure"]) 
    x["Last Name"] = x["Last Name"].astype(str)
    result = pd.merge(left = base_df, 
                  right = x, 
                  left_on = ['Arrival_Date', 'Departure_Date'], 
                  right_on =['Arrival', 'Departure'], 
                  how = 'right')
    matched_df = result[result.Reservation_No.notnull() == True]
    unmatched_df = result[result.Reservation_No.notnull() == False]
    
    base_test = matched_df["Client_Guestname_1"].str.strip()
    base_test = base_test.str.replace(',','')
    base_test = base_test.str.upper()
    
    raw_test = matched_df["Last Name"].str.strip()
    raw_test = raw_test.str.upper()
    # list conversion for fuzzy matching
    raw_list = raw_test.values.tolist()
    base_list = base_test.values.tolist()
    
    # Fuzzy matching
    possibilities = []
    for string in raw_list:
        #print(string)
        possibility = process.extractOne(string, base_list, scorer=fuzz.token_sort_ratio)
        possibilities.append(possibility)
    temp_df = pd.DataFrame(possibilities, columns = ['Match_list', 'Match_score'])
    temp_df['Actual_string'] = raw_list
    temp_df['Actual_string'] = temp_df['Actual_string'].astype(str)
    temp_df.columns = ['Match_list', 'Match_score','Actual_string']
    interm_2 = pd.merge(left = matched_df, right = temp_df, left_on = 'Last Name', right_on = 'Actual_string' , how = 'left')
    interm_3 = interm_2.append(unmatched_df) 
    threshold = 70
    interm_3['Match_status'] = interm_3['Match_score'].apply(lambda x: 'Y' if x > threshold else 'N')
    interm_3['Confirmation'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Reservation_No'], interm_3['Confirmation'])
    interm_3['Last Name'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Client_Guestname_1'], interm_3['Last Name'])
    raw_col = list(raw_df.columns)
    raw_col.append("Match_status")
    final_raw = interm_3[raw_col]
    final_raw = final_raw.drop_duplicates()
    final_raw.to_csv("C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\result_raw1.csv",mode="a",sep = ';',header=True,index=False)

reader = pd.read_csv("C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv", chunksize=1000, encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])
# reader = pd.read_csv("C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\raw_data_ex.csv",
#                      parse_dates=['Arrival','Departure','Pay Date'],
#                      chunksize=1000, 
#                      encoding = "ISO-8859-1") # chunksize depends with you colsize


start_time = time.time()
#[preprocess(r) for r in reader]
for r in reader:
    preprocess(r)
    print(r.shape)
    
end_time = time.time()
print(end_time - start_time)
# result = pd.merge(left = base_df, 
#                   right = raw_df, 
#                   left_on = ['Arrival_Date', 'Departure_Date'], 
#                   right_on =['Arrival', 'Departure'], 
#                   how = 'right')
# result
print('SUCCESS')

In [102]:
ed = pd.read_csv(r'C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv', encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date']) # chunksize depends with you colsize
ed.shape

In [81]:
## Execution

# base_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1")
# raw_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\raw_data_firstJan.csv', encoding = "ISO-8859-1")
base_df=pd.read_csv('C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv', encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])


csv_url='C:\\Users\\USER\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv'
# use chunk size 500
c_size = 2000

for gm_chunk in pd.read_csv(csv_url, encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'], chunksize=c_size):
     print(gm_chunk.shape)

base_df.shape

In [8]:
result = pd.merge(left = base_df, 
                  right = raw_df, 
                  left_on = ['Arrival_Date', 'Departure_Date'], 
                  right_on =['Arrival', 'Departure'], 
                  how = 'right',
                  indicator=True)

result.query('_merge != "both"')

In [9]:
matched_df = result[result.Reservation_No.notnull() == True]
unmatched_df = result[result.Reservation_No.notnull() == False]
matched_df

In [25]:
result1.to_excel('C:\\Users\\USER\\Documents\\misc\\test_match1.xlsx', 
                  sheet_name = 'Match',
                  header = True,
                  encoding='utf-8',
                  index=False)

In [10]:
matched_df
base_test = matched_df["Client_Guestname_1"].str.strip()
base_test = base_test.str.replace(',','')
base_test = base_test.str.upper()
#base_test = base_test.values[0]
base_test.head()

In [11]:
matched_df
raw_test = matched_df["Last Name"].str.strip()
raw_test = raw_test.str.upper()
#raw_test = str(raw_test.values[0])
raw_test.head()

In [12]:
raw_list = raw_test.values.tolist()
base_list = base_test.values.tolist()

In [13]:
#fuzz.partial_ratio(raw_test, base_test)

In [18]:
import time
possibilities = []
start_time = time.time()

for string in raw_list:
    #print(string)
    possibility = process.extractOne(string, base_list, scorer=fuzz.token_sort_ratio)
    possibilities.append(possibility)
end_time = time.time()
print(end_time - start_time)
temp_df = pd.DataFrame(possibilities)
temp_df

In [17]:
temp_df['Actual_string'] = raw_list
temp_df.columns = ['Match_list', 'Match_score', 'Actual_string']
temp_df

In [10]:
interm_2 = pd.merge(left = matched_df, right = temp_df, left_on = 'Last Name', right_on = 'Actual_string' , how = 'left')
interm_2

In [11]:
interm_3 = interm_2.append(unmatched_df) 
interm_3[['Reservation_No', 'Arrival_Date', 'Departure_Date', 'Match_score', 'Match_list']]

In [12]:

threshold = 70
interm_3['Match_status'] = interm_3['Match_score'].apply(lambda x: 'Y' if x > threshold else 'N')
interm_3['Confirmation'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Reservation_No'], interm_3['Confirmation'])
interm_3['Last Name'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Client_Guestname_1'], interm_3['Last Name'])
interm_3[['Confirmation', 'Reservation_No', 'Last Name',  'Client_Guestname_1']]
  

In [14]:
interm_3['Last name'] = np.where(interm_3['Match_status'] == 'Y', str(interm_3['Reservation_No']), str(interm_3['Last Name']))
interm_3[['Confirmation', 'Reservation_No', 'Last Name',  'Client_Guestname_1']]

In [20]:
interm_3.to_excel('C:\\Users\\USER\\Documents\\misc\\test_match_2.xlsx', 
                  sheet_name = 'Match',
                  header = True,
                  encoding='utf-8',
                  index=False)

In [16]:
matched_df.shape

In [17]:
unmatched_df.shape

In [18]:
base_df.shape

In [19]:
raw_df.shape

In [26]:
base_df.head(15)